# Enhanced Federated Fine-Tuning of Phi-4 Using OpenFL with PEFT & Quantization

In this tutorial, we demonstrate how to fine-tune Microsoft's Phi-4 model in a federated learning workflow with enhanced local training using:
- Parameter-Efficient Fine-Tuning (PEFT)
- 4-bit Quantization (QLoRA)
- Gradient Checkpointing
- Optimized Training Configuration

## Installation

In [1]:
!pip install torch transformers peft datasets trl==0.12.2 bitsandbytes accelerate -q

In [2]:
!nvidia-smi

Thu May 15 13:27:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 NVL                Off |   00000001:00:00.0 Off |                    0 |
| N/A   39C    P0             62W /  400W |       1MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Import Libraries

In [ ]:
# System imports
import os
import numpy as np

# PyTorch imports
import torch

# Hugging Face Transformers imports for model loading and training
from transformers import (
    AutoModelForCausalLM,    # For loading large language models
    AutoTokenizer,           # For tokenizing text inputs
    BitsAndBytesConfig,      # For 4-bit quantization configuration
    TrainingArguments         # For configuring training hyperparameters
)

# PEFT (Parameter-Efficient Fine-Tuning) imports
from peft import (
    LoraConfig,              # For configuring Low-Rank Adaptation
    get_peft_model,          # For applying PEFT to a model
    prepare_model_for_kbit_training,  # For preparing quantized models for training
    PeftModel                # Base class for PEFT models
)
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict  # For state dict manipulation

# Dataset and training imports
from datasets import load_dataset
from trl import SFTTrainer    # Supervised Fine-Tuning Trainer

# OpenFL imports for federated learning
from openfl.experimental.workflow.interface import Aggregator, Collaborator, FLSpec
from openfl.experimental.workflow.placement import aggregator, collaborator
from openfl.experimental.workflow.runtime import LocalRuntime

/home/azureuser/env_name/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-15 13:27:30,648	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Acquiring and preprocessing dataset

We can download the dataset directly from the [LLM-Adapters repository](https://github.com/AGI-Edgerunners/LLM-Adapters)

In [ ]:
# Import libraries needed for downloading and verifying the dataset
import hashlib
import requests

def file_checksum(file_path, algorithm="sha256"):
    """
    Calculate the checksum of a file using the specified hashing algorithm.
    
    Args:
        file_path (str): The path to the file for which the checksum is to be calculated.
        algorithm (str): The hashing algorithm to use (default is 'sha256').
    
    Returns:
        str: The calculated checksum of the file.
    """
    hash_func = hashlib.new(algorithm)
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_func.update(chunk)
    return hash_func.hexdigest()


# Download the dataset if it doesn't exist locally
if not os.path.exists("math_10k.json"):
    print("Downloading math_10k.json dataset...")
    r = requests.get(
        "https://raw.githubusercontent.com/AGI-Edgerunners/LLM-Adapters/main/ft-training_set/math_10k.json",
    )
    with open(
        "math_10k.json",
        "wb",
    ) as f:
        f.write(r.content)
    print("Download complete.")

    # Verify the integrity of the downloaded file
    actual_checksum = file_checksum("math_10k.json")
    expected_checksum = "0342d0d860ad8592b579329337c90e42eefd3d9f2898043140cbd120630418b8"
    if actual_checksum != expected_checksum:
        raise ValueError(
            "Checksum verification failed. The file may have been altered."
        )
    print("Checksum verification successful.")
else:
    print("Dataset already exists locally.")

# Set the dataset path to be used later
dataset_name = "math_10k.json"

## Configuration

In [ ]:
# Model and dataset configuration
model_name = "microsoft/phi-4"  # Pre-trained model identifier from Hugging Face Hub
#dataset_name = "math_10k.json"  # Dataset file containing mathematical QA pairs

# QLoRA (Quantized Low-Rank Adaptation) configuration for 4-bit quantization
# This reduces memory footprint while maintaining model quality
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,           # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",   # Use normalized float 4 format for better precision
    bnb_4bit_compute_dtype=torch.bfloat16,  # Computation precision
    bnb_4bit_use_double_quant=False,  # Disable nested quantization for simplicity
)

# LoRA (Low-Rank Adaptation) configuration for parameter-efficient fine-tuning
# This allows fine-tuning with significantly fewer parameters
peft_config = LoraConfig(
    r=8,                # Rank of the update matrices (higher = more capacity but more parameters)
    lora_alpha=16,      # Scaling factor for the trained weights
    lora_dropout=0.01,  # Dropout probability for LoRA layers
    bias="none",        # Don't train bias parameters to reduce memory
    task_type="CAUSAL_LM",  # Specify causal language modeling task
    target_modules="all-linear",  # Apply LoRA to all linear layers
)

# Training hyperparameters configuration
training_args = TrainingArguments(
    output_dir="./results",  # Directory to save checkpoints and logs
    num_train_epochs=1,      # Number of training epochs
    per_device_train_batch_size=2,  # Batch size per GPU/TPU core
    gradient_accumulation_steps=2,  # Number of updates steps to accumulate before backward pass
    optim="adamw_torch_fused",  # Optimizer to use (fused for better performance)
    save_steps=100,          # Save checkpoint every X updates steps
    logging_steps=10,        # Log metrics every X updates steps
    learning_rate=3e-4,      # Initial learning rate
    weight_decay=0.001,      # Weight decay regularization
    fp16=False,              # Disable FP16 training (using BF16 instead)
    bf16=True,               # Enable BF16 training (better numerical stability than FP16)
    max_grad_norm=0.5,       # Max gradient norm for gradient clipping
    warmup_ratio=0.02,       # Portion of steps for learning rate warmup
    lr_scheduler_type="cosine",  # Learning rate scheduler type
    gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
    report_to="none"         # Disable reporting to tracking platforms
)

## Load and Prepare Model

In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Apply LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:04<00:00,  1.36it/s]


trainable params: 27,852,800 || all params: 14,687,360,000 || trainable%: 0.1896


## Load and Prepare Dataset

In [ ]:
def format_prompt(example):
    """
    Format a dataset example into a standardized prompt-response format for instruction tuning.
    
    This function converts raw dataset examples into a structured format suitable for
    instruction fine-tuning of large language models. The format follows the common
    pattern used for instruction-following tasks with clear section demarcation.
    
    Args:
        example (dict): A dictionary containing the example data with keys:
            - 'instruction': The task instruction
            - 'input': The optional input context (may be empty)
            - 'output': The expected output/response
    
    Returns:
        str: A formatted prompt string with instruction, optional input, and response
    """
    if example["input"]:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

# Load dataset from JSON file (contains mathematical question-answer pairs)
dataset = load_dataset("json", data_files=dataset_name, split="train", num_proc=4)

# Transform raw examples into formatted text for instruction tuning
dataset = dataset.map(lambda x: {"text": format_prompt(x)}, num_proc=4)

# Split dataset into training (90%) and evaluation (10%) sets
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

## Enhanced Training with SFTTrainer

In [7]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=True,
)

/home/azureuser/env_name/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length, packing. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:212: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` arg

## Federated Averaging Function

In [ ]:
def FedAvg(peft_params, model, weights=None):
    """
    Perform Federated Averaging (FedAvg) on the model parameters.
    
    This function aggregates PEFT parameters from multiple collaborators using weighted
    averaging. It handles the complex task of averaging parameters while maintaining 
    the correct tensor types and shapes required by the PEFT framework.
    
    Args:
        peft_params (list): A list of state dictionaries containing PEFT parameters from different collaborators.
        model (torch.nn.Module): The base model to which the averaged parameters will be applied.
        weights (list, optional): A list of weights for averaging the parameters. If None, equal weights are used.
            Weights determine the contribution of each collaborator to the final model.
    
    Returns:
        torch.nn.Module: The model with the averaged parameters applied.
    
    Notes:
        The function converts tensors to float for averaging to avoid precision issues,
        then converts back to the original data type for model compatibility.
    """
    # Store the state dictionaries for easy access
    state_dicts = peft_params
    # Get the current state dict from the model as a template
    state_dict = get_peft_model_state_dict(model)
    
    # Iterate through each parameter in the first state dict as reference
    for key in peft_params[0]:
        # Store original data type for later conversion
        dtype = state_dicts[0][key].dtype
        
        # Convert all tensors to float, move to CPU, perform weighted average
        state_dict[key] = torch.from_numpy(
            np.average(
                [state[key].cpu().to(torch.float).numpy() for state in state_dicts], 
                axis=0, 
                weights=weights
            )
        ).to(dtype)  # Convert back to original data type
        
    # Apply the averaged parameters back to the model
    set_peft_model_state_dict(model, state_dict)
    return model

## Federated Learning Workflow

In [ ]:
# Import the required PrinterCallback for proper initialization/removal
from transformers.trainer_callback import PrinterCallback
import transformers

class FederatedFlow(FLSpec):
    """
    Federated Learning workflow for fine-tuning Phi-4 model with PEFT and quantization.
    
    This class implements the complete federated learning workflow for a language model,
    including initialization, aggregated model validation, training, local model validation,
    and parameter aggregation. It uses Parameter-Efficient Fine-Tuning (PEFT) with 4-bit
    quantization to efficiently train large language models in memory-constrained environments.
    
    The workflow follows these steps for each round:
    1. Initialize model on each collaborator
    2. Validate the aggregated model on local data
    3. Train the model locally on each collaborator
    4. Validate the locally trained model
    5. Aggregate PEFT parameters from all collaborators using FedAvg
    6. Repeat for specified number of rounds
    
    Attributes:
        model: The base language model being fine-tuned
        peft_params: PEFT parameters dictionary for the model
        optimizer: Optimizer for training (optional)
        rounds: Number of federated learning rounds to perform
        current_round: Counter for the current round
        collaborators: List of collaborators participating in federated learning
    """
    def __init__(self, model=None, optimizer=None, rounds=3, **kwargs):
        """
        Initialize the federated learning workflow.
        
        Args:
            model: The base language model to fine-tune. Must be provided.
            optimizer: Optional optimizer for model training.
            rounds: Number of federated learning rounds to perform (default: 3).
            **kwargs: Additional arguments passed to the parent class.
            
        Raises:
            ValueError: If no model is provided.
        """
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.peft_params = get_peft_model_state_dict(self.model)
            self.optimizer = optimizer
        else:
            raise ValueError("No model inputted")

        self.rounds = rounds
        

    @aggregator
    def start(self):
        """
        Start the federated learning process on the aggregator.
        
        This method initializes the workflow by:
        1. Setting up the list of collaborators from the runtime
        2. Initializing the current round counter
        3. Starting the first step of the workflow by sending the model
           to all collaborators for validation
        
        The @aggregator decorator ensures this method runs on the aggregator node.
        """
        print(f"Performing initialization for model")
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        # Start the workflow by sending the model to all collaborators
        self.next(
            self.aggregated_model_validation,
            foreach="collaborators",
        )

    
    @collaborator
    def aggregated_model_validation(self):
        """
        Validate the aggregated model on each collaborator's local dataset.
        
        This method:
        1. Loads the model with appropriate quantization configuration
        2. Applies the PEFT configuration and parameters
        3. Creates a trainer with local validation dataset
        4. Evaluates the model and records the validation loss
        5. Transitions to the training phase
        
        The @collaborator decorator ensures this method runs on each collaborator node.
        
        Notes:
            Includes fallback to CPU if GPU memory is insufficient
        """
        print(f"Performing aggregated model validation for collaborator {self.input}")
        # Load model with quantization and CPU offloading if needed
        device_map = "auto" 
        try:
            # Try to load model on GPU with quantization
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map=device_map,
                #max_memory={0: "4GiB", "cpu": "24GiB"},
                trust_remote_code=True
            )
        except ValueError:
            # Fallback to CPU if GPU memory is insufficient
            print(f"Falling back to CPU mode for {self.input}")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="cpu",
                trust_remote_code=True
            )
        
        # Prepare model for training with quantization
        self.model = prepare_model_for_kbit_training(self.model)
        # Apply PEFT configuration (LoRA)
        self.model = get_peft_model(self.model, peft_config)
        # Load aggregated parameters
        set_peft_model_state_dict(self.model, self.peft_params)
        
        # Setup trainer for evaluation
        trainer = SFTTrainer(
            model=self.model,
            args=training_args,
            peft_config=peft_config,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            max_seq_length=1024,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )

        # Remove default printer callback to avoid verbose output
        trainer.remove_callback(PrinterCallback)
        # Evaluate model and store metrics
        out = trainer.evaluate()
        self.agg_validation_score = out["eval_loss"]
        print(f"{self.input} value of {self.agg_validation_score}")
        # Move to training phase
        self.next(self.train)

    @collaborator
    def train(self):
        """
        Train the model on each collaborator's local dataset.
        
        This method:
        1. Creates an SFTTrainer with the local training dataset
        2. Runs the training process
        3. Records the training loss
        4. Saves the trained model
        5. Transitions to local validation phase
        
        The @collaborator decorator ensures this method runs on each collaborator node.
        """
        # Setup trainer for local training
        trainer = SFTTrainer(
            model=self.model,
            args=training_args,
            peft_config=peft_config,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            max_seq_length=1024,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )

        # Execute training
        out = trainer.train()
        # Store training loss for later analysis
        self.loss = out.training_loss
        # Save locally trained model
        trainer.save_model()
        self.training_completed = True
        # Move to local validation phase
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        """
        Validate the locally trained model on each collaborator's validation dataset.
        
        This method:
        1. Creates an SFTTrainer with the local validation dataset
        2. Evaluates the locally trained model
        3. Records the validation loss
        4. Extracts the PEFT parameters for aggregation
        5. Sends results to the aggregator for parameter aggregation
        
        The @collaborator decorator ensures this method runs on each collaborator node.
        
        Notes:
            Excludes the full model and training flags from the data sent to the aggregator
            to reduce communication overhead
        """
        # Setup trainer for evaluation
        trainer = SFTTrainer(
            model=self.model,
            args=training_args,
            peft_config=peft_config,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            max_seq_length=1024,
            dataset_text_field="text",
            tokenizer=tokenizer,
            packing=True,
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )
        # Evaluate the locally trained model
        out = trainer.evaluate()
        self.local_validation_score = out["eval_loss"]
        # Extract PEFT parameters for aggregation
        self.peft_params = get_peft_model_state_dict(self.model)
        print(f"Doing local model validation for collaborator {self.input}")
        # Send results to aggregator, excluding the full model and training flags
        # to reduce communication overhead
        self.next(self.join, exclude=["training_completed", "model"])

    @aggregator
    def join(self, inputs):
        """
        Aggregate results from all collaborators and update the global model.
        
        This method:
        1. Calculates average loss, aggregated model accuracy, and local model accuracy
        2. Updates the global model using Federated Averaging (FedAvg)
        3. Saves the aggregated model and tokenizer
        4. Either starts the next round or ends the workflow depending on round count
        
        Args:
            inputs: List of data objects from all collaborators containing validation scores
                   and PEFT parameters.
            
        The @aggregator decorator ensures this method runs on the aggregator node.
        """
        # Calculate average metrics across all collaborators
        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs
        ) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs
        ) / len(inputs)
        
        # Display aggregated metrics
        print(
            f"Average aggregated model validation values = {self.aggregated_model_accuracy}"
        )
        print(f"Average training loss = {self.average_loss}")
        print(f"Average local model validation values = {self.local_model_accuracy}")

        # Perform federated averaging of model parameters
        self.model = FedAvg([input.peft_params for input in inputs], self.model)
        self.peft_params = get_peft_model_state_dict(self.model)

        # Save the aggregated model for future use
        self.model.save_pretrained("./aggregated/model")
        tokenizer.save_pretrained("./aggregated/tokenizer")
        
        # Increment round counter and start next round or end workflow
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(
                self.aggregated_model_validation,
                foreach="collaborators",
                exclude=["model"],
            )
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        """
        End the federated learning process.
        
        This method marks the end of the federated learning workflow after all rounds
        have been completed. The final aggregated model and tokenizer are already saved
        in the last join step.
        
        The @aggregator decorator ensures this method runs on the aggregator node.
        """
        print(f"This is the end of the flow")

Aggregator step "start" registered
Collaborator step "aggregated_model_validation" registered
Collaborator step "train" registered
Collaborator step "local_model_validation" registered
Aggregator step "join" registered
Aggregator step "end" registered


## Run Federated Learning

In [ ]:
# Setup federated learning participants
aggregator = Aggregator()  # Central coordinator that aggregates model updates
collaborators = [
    Collaborator(name="Portland"),  # First participant with local dataset
    Collaborator(name="Seattle")    # Second participant with local dataset
]

# Distribute data shards to collaborators (simulating data silos)
# Each collaborator gets a non-overlapping portion of the dataset
for idx, colab in enumerate(collaborators):
    colab.private_attributes = {
        "train_dataset": train_dataset.shard(len(collaborators), idx),  # Training shard
        "eval_dataset": eval_dataset.shard(len(collaborators), idx)     # Evaluation shard
    }

# Set up and execute the federated learning workflow
runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators)  # Local simulation runtime
flflow = FederatedFlow(model, rounds=2)  # Create flow with 2 federated learning rounds
flflow.runtime = runtime  # Assign runtime to the flow
flflow.run()  # Start the federated learning process


Calling start
Performing initialization for model

Calling aggregated_model_validation
Performing aggregated model validation for collaborator Portland


Loading checkpoint shards: 100%|##########| 6/6 [00:04<00:00,  1.30it/s]
/home/azureuser/env_name/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field, packing. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:212: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/train

Portland value of 0.5918120741844177

Calling train


/home/azureuser/env_name/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field, packing. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:212: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` arg

Step,Training Loss
10,0.516900
20,0.373400
30,0.346100
40,0.339100
50,0.333000
60,0.323700
70,0.329800
80,0.312800
90,0.326000
100,0.306800


...

Doing local model validation for collaborator Portland
Should transfer from local_model_validation to join

Calling aggregated_model_validation
Performing aggregated model validation for collaborator Seattle


Loading checkpoint shards: 100%|##########| 6/6 [00:04<00:00,  1.30it/s]
/home/azureuser/env_name/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field, packing. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:212: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/train

Seattle value of 0.31057578325271606

Calling train


/home/azureuser/env_name/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field, packing. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:212: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/azureuser/env_name/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` arg

Step,Training Loss
10,0.300900
20,0.307900
30,0.303400
40,0.274300
50,0.295200
60,0.270700
70,0.280300
80,0.284600
90,0.298700
100,0.290900
